# 01. 이미지 Segmentation

In [ ]:
# import module & model

from transformers import SegformerImageProcessor, AutoModelForSemanticSegmentation
from PIL import Image
import requests
import matplotlib.pyplot as plt
import torch.nn as nn
import torch
import torchvision.transforms as transforms
import numpy as np
import cv2

processor = SegformerImageProcessor.from_pretrained("mattmdjaga/segformer_b2_clothes")
model = AutoModelForSemanticSegmentation.from_pretrained("mattmdjaga/segformer_b2_clothes")

import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'

In [ ]:
# image segmentation

    ### url로 불러오기 ###
url = 'https://image.msscdn.net/thumbnails/display/images/usersnap/2024/02/09/f5344ae7c72040a78dc6c10170843764.jpg?w=3900'
image = Image.open(requests.get(url, stream=True).raw)


    ### 이미지 파일로 불러오기 ###
# image_folder = './segmentation_comparison1.jpg'
# image = Image.open(image_folder)


inputs = processor(images=image, return_tensors="pt")

outputs = model(**inputs)
logits = outputs.logits.cpu()

upsampled_logits = nn.functional.interpolate(
    logits,
    size=image.size[::-1],
    mode="bilinear",
    align_corners=False,
)

pred_seg = upsampled_logits.argmax(dim=1)[0]
plt.imshow(pred_seg)

In [ ]:
# label명 확인
model.config.id2label

In [ ]:
# color map 생성

import matplotlib as mpl
label_names = list(model.config.id2label)
# Create a color map with the same number of colors as your labels
# Use the updated method to get the colormap
cmap = mpl.colormaps['tab20']

# Create the figure and axes for the plot and the colorbar
fig, ax = plt.subplots()

# Display the segmentation
im = ax.imshow(pred_seg, cmap=cmap)

# Create a colorbar
cbar = fig.colorbar(im, ax=ax, ticks=range(len(label_names)))
cbar.ax.set_yticklabels(label_names)

plt.show()

# Get the number of labels
n_labels = len(label_names)

# Extract RGB values for each color in the colormap
colors = cmap.colors[:n_labels]

# Convert RGBA to RGB by omitting the Alpha value
rgb_colors = [color[:3] for color in colors]

# Create a dictionary mapping labels to RGB colors
label_to_color = dict(zip(label_names, rgb_colors))

# Display the mapping
for label, color in label_to_color.items():
    print(f"{label}: {color}")

In [ ]:
# class별 이미지 시각화

segments = torch.unique(pred_seg) # Get a list of all the predicted items
for i in segments:
    mask = pred_seg == i # Filter out anything that isn't the current item
    img = Image.fromarray((mask * 255).numpy().astype(np.uint8))
    name = model.config.id2label[i.item()] # get the item name
    plt.imshow(img)
    plt.title(name)
    plt.show()

In [ ]:
# 원본 이미지에서 추출

image = np.array(image)
for i in segments:
# 마스크 생성
  desired_label = i
  label_name = model.config.id2label[i.item()]
  mask = pred_seg == desired_label

  if mask.sum() <= 3000:
    continue
  elif i.item() in [0, 2, 11, 12, 13, 14, 15]: ## 얼굴 팔 머리 같은 것들이라 저장 x
    continue
  else:
    # 이미지와 마스크를 비트와이즈 연산하여 추출된 부분을 얻음
    mask_np = mask.numpy().astype(np.uint8) * 255
    result = cv2.bitwise_and(image.astype(np.uint8), image.astype(np.uint8), mask=mask_np)

    # 결과 이미지를 화면에 표시하거나 저장
    plt.imshow(result)
    plt.axis('off')
    plt.show()

    # 결과 이미지 저장
    result_rgb = cv2.cvtColor(result, cv2.COLOR_BGR2RGB)
    cv2.imwrite(f'{label_name}.jpg', result_rgb)

---

# 02. Object Detection + Cropping

In [ ]:
# Load model directly
from transformers import AutoImageProcessor, AutoModelForObjectDetection

processor = AutoImageProcessor.from_pretrained("chiHang/detr-resnet50-finetuned_clothes0")
model = AutoModelForObjectDetection.from_pretrained("chiHang/detr-resnet50-finetuned_clothes0")

In [ ]:
from transformers import AutoImageProcessor, DetrForObjectDetection

# url = "https://image.msscdn.net/thumbnails/display/images/usersnap/2024/02/09/f5344ae7c72040a78dc6c10170843764.jpg?w=390"
# image = Image.open(requests.get(url, stream=True).raw)

image_folder = './sample_pants.jpg'
image = Image.open(image_folder)


image_processor = AutoImageProcessor.from_pretrained("facebook/detr-resnet-50")
model = DetrForObjectDetection.from_pretrained("facebook/detr-resnet-50")

inputs = image_processor(images=image, return_tensors="pt")
outputs = model(**inputs)

# convert outputs (bounding boxes and class logits) to Pascal VOC format (xmin, ymin, xmax, ymax)
target_sizes = torch.tensor([image.size[::-1]])
results = image_processor.post_process_object_detection(outputs, threshold=0.9, target_sizes=target_sizes)[
    0
]

for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
    box = [round(i, 2) for i in box.tolist()]
    print(
        f"Detected {model.config.id2label[label.item()]} with confidence "
        f"{round(score.item(), 3)} at location {box}"
    )

In [ ]:
# Open the image
image_folder = './sample_pants.jpg'
image = Image.open(image_folder)

# image = Image.open(requests.get(url, stream=True).raw)

# Plot the image
plt.imshow(image)
plt.axis('off')

# Draw bounding boxes around detected objects
for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
    box = [round(i, 2) for i in box.tolist()]
    xmin, ymin, xmax, ymax = box
    plt.plot([xmin, xmax, xmax, xmin, xmin], [ymin, ymin, ymax, ymax, ymin], linewidth=2, label=model.config.id2label[label.item()] + f' {score.item():.2f}')

# Show the plot
plt.show()


In [ ]:
# Crop and display detected objects
for idx, (score, label, box) in enumerate(zip(results["scores"], results["labels"], results["boxes"])):
    box = [round(i, 2) for i in box.tolist()]
    xmin, ymin, xmax, ymax = box
    roi = image.crop((xmin, ymin, xmax, ymax))
    plt.imshow(roi)
    plt.axis('off')
plt.show()

In [ ]:
# Create a directory to save cropped images
output_dir = "./cropped_images"
os.makedirs(output_dir, exist_ok=True)

# Loop through each detected object
for idx, (score, label, box) in enumerate(zip(results["scores"], results["labels"], results["boxes"])):
    box = [round(i, 2) for i in box.tolist()]  # Round the box coordinates
    xmin, ymin, xmax, ymax = box
    
    # Crop the detected object from the original image
    roi = image.crop((xmin, ymin, xmax, ymax))
    
    # Save the cropped image
    cropped_image_path = os.path.join(output_dir, f"cropped_object_0.jpg")
    roi.save(cropped_image_path)

---

# 03. 이미지 벡터화

In [ ]:
# 이미지 벡터화 함수정의
def image_to_vector(image_path, output_path, resize_size=(256, 256)):  # 이미지 size 변환 resize(256,256)
    image = Image.open(image_path)
    image = image.resize(resize_size)
    image_array = np.array(image, dtype=np.float32)
    image_vector = image_array.flatten()
    np.savetxt(output_path, image_vector)

In [ ]:
image_to_vector('./cropped_images/cropped_object_0.jpg', './cropped_vectors/cropped_vectors_0.txt', resize_size=(256, 256))

image_to_vector('./cropped_images/cropped_object_1.jpg', './cropped_vectors/cropped_vectors_1.txt', resize_size=(256, 256))
image_to_vector('./cropped_images/cropped_object_2.jpg', './cropped_vectors/cropped_vectors_2.txt', resize_size=(256, 256))
image_to_vector('./cropped_images/cropped_object_3.jpg', './cropped_vectors/cropped_vectors_3.txt', resize_size=(256, 256))
image_to_vector('./cropped_images/cropped_object_4.jpg', './cropped_vectors/cropped_vectors_4.txt', resize_size=(256, 256))

In [ ]:
cropped_vectors_0 = './cropped_vectors/cropped_vectors_0.txt'

cropped_vectors_1 = './cropped_vectors/cropped_vectors_1.txt'
cropped_vectors_2 = './cropped_vectors/cropped_vectors_2.txt'
cropped_vectors_3 = './cropped_vectors/cropped_vectors_3.txt'
cropped_vectors_4 = './cropped_vectors/cropped_vectors_4.txt'

---

# 04. 유사도 분석 (cosine_similarity)

In [ ]:
## 유사도 분석 함수정의
def cosine_similarity(vec1_path, vec2_path):
    vec1 = np.loadtxt(vec1_path)
    vec2 = np.loadtxt(vec2_path)
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    similarity = dot_product / (norm_vec1 * norm_vec2)
    
    return similarity

In [ ]:
similarity1 = cosine_similarity(cropped_vectors_0, cropped_vectors_1)
similarity2 = cosine_similarity(cropped_vectors_0, cropped_vectors_2)
similarity3 = cosine_similarity(cropped_vectors_0, cropped_vectors_3)
similarity4 = cosine_similarity(cropped_vectors_0, cropped_vectors_4)

print("cropped_vectors_1과의 유사도:", similarity1)
print("cropped_vectors_2과의 유사도:", similarity2)
print("cropped_vectors_3과의 유사도:", similarity3)
print("cropped_vectors_4과의 유사도:", similarity4)